<a href="https://colab.research.google.com/github/DanButterfield/Fine-Tuning-Qwen/blob/main/data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install torch

In [ ]:
import shutil
shutil.rmtree("/root/.cache/huggingface/datasets", ignore_errors=True)

In [ ]:
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling

import os
import copy
import gc
import traceback
import torch
from torch.utils.data import DataLoader
from bitsandbytes.optim import PagedAdamW8bit # Better optimiser

from google.colab import drive
drive.mount('/content/drive')

from tqdm import tqdm # Progress bar for training
import matplotlib.pyplot as plt # Plotting

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load and Split Dataset

# Load Model

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct" # for a comparison

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

# Tokenisation

In [ ]:
def tokenise_convos(example, tokenizer):
  role_map = {"system": "system", "human": "user", "gpt": "assistant"}
  converted = []
  for turn in example["conversations"]:
    role = role_map.get(turn["from"])
    if role:
      converted.append({"role": role, "content": turn["value"]})
      role = role_map.get(turn["from"])
      if role:
        converted.append({"role": role, "content": turn["value"]})

  formatted_text = tokenizer.apply_chat_template(converted, tokenize=False)

  tokenised = tokenizer(
    formatted_text,
    truncation=True,
    max_length=256,
    padding="max_length"
  )

  labels = tokenised["input_ids"].copy()
  labels = [-100 if token == tokenizer.pad_token_id else token for token in labels]
  tokenised["labels"] = labels
  return tokenised

In [ ]:
import shutil
shutil.rmtree("/root/.cache/huggingface/datasets", ignore_errors=True)

full_dataset = load_dataset("teknium/OpenHermes-2.5", split="train[:10000]")

def is_valid_conversation(example):
  roles = [c["from"] for c in example["conversations"]]
  return "human" in roles and "gpt" in roles

filtered = full_dataset.filter(
  is_valid_conversation,
  load_from_cache_file=False)
print(f"After filter: {len(filtered)}")

tokenised_ds = filtered.map(
  tokenise_convos,
  batched = False,
  remove_columns=filtered.column_names,
  fn_kwargs={"tokenizer": tokenizer},
  load_from_cache_file=False
)
print(f"After tokenisation: {len(tokenised_ds)}")

Generating train split:   0%|          | 0/1001551 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

After filter: 10000


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

After tokenisation: 10000


# Process Data

Split data into 90/10 split for training and validation

In [ ]:
def collatefn(batch):
  return {
      key: torch.stack([torch.tensor(example[key]) for example in batch])
      for key in batch[0].keys()
  }

In [ ]:
dataset_splits = tokenised_ds.train_test_split(test_size=0.2, seed=42)
train_data = dataset_splits["train"]
val_data = dataset_splits["test"]


BATCH_SIZE = 1

train_loader = DataLoader(
    train_data,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collatefn
)

val_loader = DataLoader(
    val_data,
    batch_size = BATCH_SIZE,
    shuffle = False,
    collate_fn = collatefn
)

# Training Loop

In [ ]:
def train_llm(model, train_loader, val_loader, device, model_name, num_epochs=3, patience=3, lr=2e-5, weight_decay=0.01):
  net = model.to(device)
  optimiser = PagedAdamW8bit(net.parameters(), lr=lr, weight_decay=weight_decay)

  best_val_loss = float('inf')
  patience_counter = 0
  best_model_wts = copy.deepcopy(net.state_dict())
  train_losses, val_losses = [], []

  for epoch in range(num_epochs):
    # training
    net.train()
    running_loss = 0.0
    prog_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}")
    for i, data in prog_bar:
      batch = {k: v.to(device) for k, v in data.items()}
      optimiser.zero_grad()


      outputs = net(**batch)
      loss = outputs.loss

      loss.backward()
      optimiser.step()
      running_loss += loss.item()
      prog_bar.set_postfix({"loss": running_loss / (i + 1)})

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # validation
    net.eval()
    val_running_loss = 0.0
    with torch.no_grad():
      for i, data in enumerate(val_loader):
        batch = {k: v.to(device) for k, v in data.items()}
        outputs = net(**batch)
        loss = outputs.loss
        val_running_loss += loss.item()

    val_loss = val_running_loss / len(val_loader)
    val_losses.append(val_loss)

    print(f'Epoch {epoch +1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}')

    # Early Stopping
    if val_loss < best_val_loss:
      best_val_loss = val_loss
      best_model_wts = copy.deepcopy(net.state_dict())
      patience_counter = 0
    else:
      patience_counter += 1
      if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        break

    net.load_state_dict(best_model_wts)
    model_path = f"/content/drive/MyDrive/{model_name}.pth"
    torch.save(net.state_dict(), model_path)
    print(f"[{model_name}] Best chatbot adapter saved as {model_path}")
    return net, train_losses, val_losses

In [ ]:
gc.collect()
torch.cuda.empty_cache()
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype = torch.bfloat16 )



trained_model, train_loss, val_loss = train_llm(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    model_name = "First-GPT",
    num_epochs=3,
    patience=1,
    lr=2e-5
)


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Epoch 1: 100%|██████████| 8000/8000 [27:39<00:00,  4.82it/s, loss=0.626]


Epoch 1: Train Loss = 0.6262, Val Loss = 0.5947
[First-GPT] Best chatbot adapter saved as /content/drive/MyDrive/First-GPT.pth


In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/First-GPT.pth"))
model.eval() # call model from drive

base_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16).to(device) # base mdoel for a comparison on behaviour

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
def gen_response(user_input, model, tokenizer, device):
  messages = [{"role": "user", "content": user_input}]
  formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(formatted, return_tensors="pt").to(device)
  outputs = model.generate(**inputs, max_new_tokens=200)
  full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
  # strip everything up to and including "assistant\n"
  if "assistant\n" in full_response:
      full_response = full_response.split("assistant\n")[-1]
  return full_response.strip()

In [ ]:
print("Assistant: Hi! How can I help?")

while True:
  user_input = input("You: ")

  if user_input.lower() in ["quit", "exit", "goodbye", "bye"]:
    print("Assistant: See You Soon!")
    break

  #response = gen_response(user_input, trained_model, tokenizer, device)
  response = gen_response(user_input, base_model, tokenizer, device)
  print("Assistant:", response)
  results = compute_perplexity(user_input, response, model, tokenizer, device)
  print(results)

Assistant: Hi! How can I help?
You: what is 47*83
Assistant: To calculate \( 47 \times 83 \), you can use the standard multiplication method:

1) Break down the numbers into tens and ones:
   - \( 47 = 40 + 7 \)
   - \( 83 = 80 + 3 \)

2) Multiply each part of one number by each part of the other number:
   - \( 40 \times 80 = 3200 \)
   - \( 40 \times 3 = 120 \)
   - \( 7 \times 80 = 560 \)
   - \( 7 \times 3 = 21 \)

3) Add all these products together:
   - \( 3200 + 120 + 560 + 21 = 3901 \)

So, \( 47 \times 83 = 390
{'perplexities': [1.3984375], 'mean_perplexity': 1.3984375}
You: solve 2x+5=13
Assistant: To solve the equation \(2x + 5 = 13\), follow these steps:

1. Subtract 5 from both sides of the equation to isolate the term with the variable:
   \[
   2x + 5 - 5 = 13 - 5
   \]
   Simplifying this gives:
   \[
   2x = 8
   \]

2. Divide both sides by 2 to solve for \(x\):
   \[
   \frac{2x}{2} = \frac{8}{2}
   \]
   Simplifying this gives:
   \[
   x = 4
   \]

So, the solution 

In [ ]:
print("Assistant: Hi! How can I help?")

while True:
  user_input = input("You: ")

  if user_input.lower() in ["quit", "exit", "goodbye", "bye"]:
    print("Assistant: See You Soon!")
    break

  response = gen_response(user_input, model, tokenizer, device)
  #response = gen_response(user_input, base_model, tokenizer, device)
  print("Assistant:", response)
  results = compute_perplexity(user_input, response, model, tokenizer, device)
  print(results)

Assistant: Hi! How can I help?
You: what is 47*83
Assistant: The product of 47 and 83 is 3911.
{'perplexities': [1.2734375], 'mean_perplexity': 1.2734375}
You: solve 2x+5=13
Assistant: Sure! Here's how you can solve it:

Given the equation:
2x + 5 = 13

First, subtract 5 from both sides of the equation to isolate terms with x on one side and constants on the other:

2x + 5 - 5 = 13 - 5

This simplifies to:

2x = 8

Next, divide both sides by 2 to find the value of x:

(2x) / 2 = 8 / 2

This simplifies to:

x = 4

So, the solution to the equation 2x + 5 = 13 is x = 4.
{'perplexities': [1.2109375], 'mean_perplexity': 1.2109375}
You: factor x^2+10x+25
Assistant: The expression can be factored as (x + 5)^2.
{'perplexities': [1.421875], 'mean_perplexity': 1.421875}
You: bye
Assistant: See You Soon!


# Metrics:

## Perplexity
Perplexity measures a models uncertainty, specifically how well a model can predict the next word.

Calculated via P=2^H(p)

With H(p) as the models entropy, a lower entropy value means a model is more certain about its predictions. In general a lower perplexity is preferred as that indicates a model is more confident in its predictions.

Over a batch of words the formula looks like:

$$
p = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log p(w_i \mid w_{i-1}, w_{i-2}, \dots, w_1)\right)
$$

where, p(...) is the predicted probability of the i^th word and N is the words in the sequence.
### Why is it important?
Perplexity gives: Prediction accuracy, model confidence and a way to evaluate language models.

However, sometimes models can have a low perplexity, but still provide the wrong answers.

## ROUGE

In [ ]:
# Adapted from: GeeksforGeeks - Perplexity for LLM Evaluation
# https://www.geeksforgeeks.org/nlp/perplexity-for-llm-evaluation/
def compute_perplexity(user_input, assistant_response, model, tokenizer, device):
  # Build the full conversation including the response
  messages = [
    {"role": "user", "content": user_input},
    {"role": "assistant", "content": assistant_response}
  ]

  # Prompt-only length for masking
  prompt_only = tokenizer.apply_chat_template(
    [{"role": "user", "content": user_input}],
    tokenize=False, add_generation_prompt=True
  )
  prompt_len = tokenizer(prompt_only, return_tensors="pt")["input_ids"].shape[1]

  # Full conversation
  full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
  inputs = tokenizer(full_text, return_tensors="pt").to(device)
  input_ids = inputs["input_ids"]
  attention_mask = inputs["attention_mask"]

  with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits

  shift_logits = logits[:, :-1, :]
  shift_labels = input_ids[:, 1:]

  log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
  target_log_probs = log_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)

  completion_mask = attention_mask[:, 1:].clone().to(log_probs.dtype)
  completion_mask[:, :prompt_len - 1] = 0

  target_log_probs = target_log_probs * completion_mask
  negative_log_likelihood = -target_log_probs.sum(dim=-1) / completion_mask.sum(dim=-1)
  perplexities = torch.exp(negative_log_likelihood)

  return {
    "perplexities": perplexities.tolist(),
    "mean_perplexity": perplexities.mean().item()
  }

get it working

add context



